In [ ]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import WeightedRandomSampler
import torch.nn as nn
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from PIL import Image
import os
from pathlib import Path
import kagglehub, pathlib
import pickle
import random
import warnings
warnings.filterwarnings('ignore')

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [ ]:
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
drive.mount('/content/drive')
storage_root = Path('/content/drive/MyDrive')
print('Гугл диск подключён')
except ImportError:
    storage_root = Path.cwd()
    print('Нет гугла, работает черещ локалку')

In [ ]:
project_dir = storage_root / 'GP5_DL_image_classification'
mlflow_dir = project_dir / 'mlflow'
mlflow_artifacts_dir = mlflow_dir / 'artifacts'
checkpoints_dir = project_dir / 'checkpoints'
predictions_dir = project_dir / 'predictions'
reports_dir = project_dir / 'reports'
splits_dir = project_dir / 'splits'
preprocessing_dir = project_dir / 'preprocessing'

In [ ]:
for folder in [project_dir, mlflow_dir, mlflow_artifacts_dir, checkpoints_dir, predictions_dir, reports_dir, splits_dir, preprocessing_dir]:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
mlflow_db_path = mlflow_dir / 'mlflow.db'
label_encoder_path = preprocessing_dir / 'label_encoder.pkl'
split_manifest_path = splits_dir / 'train_val_split.csv'

In [ ]:
print('Папка проекта:', project_dir)

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print('Training on GPU')
else:
    DEVICE = torch.device('cpu')
    print('Training on CPU')

In [ ]:
ds_root = pathlib.Path(kagglehub.dataset_download('stepanyarullin/interior-design-styles'))
print('Kaggle Hub cached at:', ds_root)

In [ ]:
dataset_dir = Path(ds_root)
print('Папка датасета', dataset_dir)

In [ ]:
train_dir = dataset_dir / 'dataset_train'
test_dir = dataset_dir / 'dataset_test'
test_csv = dataset_dir / 'test_labels.csv'

In [ ]:
assert train_dir.exists(), f'Не найдена папка {train_dir}'
assert test_dir.exists(), f'Не найдена папка {test_dir}'
assert test_csv.exists(), f'Не найден файл с тестовыми метками {test_csv}'

In [ ]:
rescale_size = 224

In [ ]:
print(f'Устройство для обучения: {DEVICE}')

In [ ]:
class InteriorDataset(Dataset):
    def __init__(self, files, mode='train', label_encoder=None):
        super().__init__()
        self.files = sorted(files)
        self.mode = mode
        if mode != 'test':
            self.labels = [path.parent.name for path in self.files]
            if label_encoder is None:
                self.label_encoder = LabelEncoder()
                self.label_encoder.fit(self.labels)
            else:
                self.label_encoder = label_encoder

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        image = Image.open(self.files[index]).convert('RGB')
        transform = transforms.Compose([
            transforms.Resize((rescale_size, rescale_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        if self.mode == 'train':
            transform = transforms.Compose([
                transforms.Resize((rescale_size, rescale_size)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
        image = transform(image)
        if self.mode == 'test':
            return image
        label = self.labels[index]
        label_id = self.label_encoder.transform([label])[0]
        return image, label_id

In [ ]:
train_val_files = list(Path(train_dir).rglob('*.jpg'))
test_files = list(Path(test_dir).rglob('*.jpg'))

In [ ]:
len(train_val_files)

In [ ]:
Всего картинок в train+val 14876

In [ ]:
len(test_files)

In [ ]:
Всего картинок в test 3729

In [ ]:
all_labels = [p.parent.name for p in train_val_files]
label_encoder = LabelEncoder()
label_encoder.fit(all_labels)

In [ ]:
len(label_encoder.classes_)

In [ ]:
Всего стилей: 19

In [ ]:
list(label_encoder.classes_)

In [ ]:
Стили: [np.str_('asian'), np.str_('coastal'), np.str_('contemporary'), np.str_('craftsman'), np.str_('eclectic'), np.str_('farmhouse'), np.str_('french-country'), np.str_('industrial'), np.str_('mediterranean'), np.str_('mid-century-modern'), np.str_('modern'), np.str_('rustic'), np.str_('scandinavian'), np.str_('shabby-chic-style'), np.str_('southwestern'), np.str_('traditional'), np.str_('transitional'), np.str_('tropical'), np.str_('victorian')]


In [ ]:
train_files, val_files = train_test_split(train_val_files, test_size=0.2, stratify=all_labels, random_state=42)

In [ ]:
len(train_files)

In [ ]:
len(val_files)

In [ ]:
train_dataset = InteriorDataset(train_files, mode='train', label_encoder=label_encoder)
val_dataset = InteriorDataset(val_files, mode='val', label_encoder=label_encoder)
test_dataset = InteriorDataset(test_files, mode='test')

In [ ]:
test_labels_for_manifest = pd.read_csv(test_csv)
test_style_by_filename = dict(zip(test_labels_for_manifest['filename'], test_labels_for_manifest['style']))

In [ ]:
split_manifest_df = pd.DataFrame({
    'filepath': [str(path) for path in train_files] + [str(path) for path in val_files] + [str(path) for path in test_files],
    'filename': [path.name for path in train_files] + [path.name for path in val_files] + [path.name for path in test_files],
    'style': [path.parent.name for path in train_files] + [path.parent.name for path in val_files] + [test_style_by_filename.get(path.name, 'unknown') for path in test_files],
    'split': ['train'] * len(train_files) + ['validation'] * len(val_files) + ['test'] * len(test_files),
    'dataset_source': ['stepanyarullin/interior-design-styles'] * (len(train_files) + len(val_files) + len(test_files))
})

In [ ]:
split_manifest_df.to_csv(split_manifest_path, index=False, encoding='utf-8')
print('Разделили train/validation/test и сохранили:', split_manifest_path)
display(split_manifest_df['split'].value_counts())

In [ ]:
display(split_manifest_df.groupby(['split', 'style']).size().unstack(fill_value=0).T.head())

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
axes = axes.flatten()

for i in range(10):
    random_idx = np.random.randint(0, len(val_dataset))
    img, label = val_dataset[random_idx]
    img_display = img.cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_display = std * img_display + mean
    img_display = np.clip(img_display, 0, 1)
    style_name = label_encoder.inverse_transform([label])[0]
    axes[i].imshow(img_display)
    axes[i].set_title(f"Стиль: {style_name}")
    axes[i].axis('off')
plt.suptitle("Интерьерные стили")
plt.show()

In [ ]:
class_counts = Counter(all_labels)

plt.figure(figsize=(12, 5))
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()))
plt.xticks(rotation=90)
plt.title('Количество картинок по стилям')
plt.ylabel('Количество')
plt.show()

In [ ]:
for style, count in class_counts.most_common():
    print(f"{style:25} | {count:4} картинок")

In [ ]:
with open(label_encoder_path, 'wb') as f:
    pickle.dump(train_dataset.label_encoder, f)
print('Label encoder сохранен:', label_encoder_path)